In [ ]:
# Remember to make a venv and install the dependencies before running this notebook. You can do this with pip install -r requirements.txt if you have a requirements.txt file with the necessary dependencies, 
# py -m venv venv && source venv/bin/activate && pip install -r requirements.txt

# Or you can run the following commands to install them directly (after making your venv and activating it):

!pip install git+https://github.com/unslothai/unsloth-zoo.git git+https://github.com/unslothai/unsloth.git pandas numpy torch pathlib kaggle trl==0.24.0 bitsandbytes picosvg cairosvg opencv-python-headless editdistance mergekit llm_blender outlines scikit-learn weave wandb protobuf llguidance

# !pip install -r requirements.txt

In [ ]:
# !mkdir -p ~/.kaggle
# !mv kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

In [1]:
import os
import pandas as pd
from pathlib import Path
import numpy as np
import torch

In [ ]:
import os
import pandas as pd
from pathlib import Path
import numpy as np
import torch

BASE_DIR = Path("./dl-spring-2026-svg-generation")

# Download data
if not BASE_DIR.exists() or not any(BASE_DIR.iterdir()):
    print("Downloading and unzipping data...")
    os.makedirs(BASE_DIR, exist_ok=True)
    print("Downloading...")
    !kaggle competitions download -c dl-spring-2026-svg-generation
    print("Unzipping...")
    !unzip -q dl-spring-2026-svg-generation.zip
    !mv sample_submission.csv /content/dl-spring-2026-svg-generation
    !mv train.csv /content/dl-spring-2026-svg-generation
    !mv test.csv /content/dl-spring-2026-svg-generation
else:
    print("SVG Data folder already exists.")

SVG Data folder already exists.


In [3]:
import pandas as pd
import concurrent.futures
from picosvg.svg import SVG
import re
import os
import numpy as np

FLOAT_PATTERN = re.compile(r'-?\d+\.\d+')

def round_floats(match):
    return f"{float(match.group(0)):.2f}".rstrip('0').rstrip('.')

def optimize_svg(raw_svg):
    if not isinstance(raw_svg, str) or not raw_svg.strip(): return ""
    try:
        svg_obj = SVG.fromstring(raw_svg)
        cleaned_svg = svg_obj.topicosvg().tostring()
        return FLOAT_PATTERN.sub(round_floats, cleaned_svg)
    except Exception:
        return ""

def process_chunk(chunk):
    chunk['svg'] = chunk['svg'].apply(optimize_svg)
    return chunk

# Load, parallel process, filter, and save (Comment out if you already have the optimized file)
# df = pd.read_csv("dl-spring-2026-svg-generation/train.csv")
# num_cores = os.cpu_count() or 4
# chunks = np.array_split(df, num_cores * 4)

# processed_chunks = []
# with concurrent.futures.ProcessPoolExecutor(max_workers=num_cores) as executor:
#     for result in executor.map(process_chunk, chunks):
#         processed_chunks.append(result)

# final_df = pd.concat(processed_chunks)
# final_df = final_df[final_df['svg'] != ""]
# final_df.to_parquet("train_optimized.parquet", index=False)
# print(f"Data Diet Complete. {len(final_df)} rows saved.")

In [ ]:
import os
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import torch

# Load Dataset and apply the 25,000 row limit for the 12-hour Kaggle window
raw_dataset = load_dataset("parquet", data_files="./train_optimized.parquet", split="train")
sliced_dataset = raw_dataset.select(range(25000))

MODEL_ID = "unsloth/Qwen2.5-Coder-3B"

# Lower max_seq_length to 1536 to protect VRAM for the 3B model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=1536,
    # dtype=torch.float16, # Explicitly use float16 for T4s
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, 
    r=16, 
    lora_alpha=32, 
    lora_dropout=0, 
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], 
    use_gradient_checkpointing="unsloth", # Crucial for 3B
    random_state=42,
)

EOS = tokenizer.eos_token

# The Conditional Format: Teaches the model to read the prompt!
def format_for_sft(example):
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"
    
    text = f"{open_comment} Description: {example['prompt']} {close_comment}\n```xml\n{example['svg']}\n```{EOS}"
    return {"text": text}

sft_dataset = sliced_dataset.map(format_for_sft)

# 3B-Optimized SFT Config
sft_config = SFTConfig(
    output_dir="./svg-phase1-sft",
    dataset_text_field="text",
    max_seq_length=1536,
    num_train_epochs=1, 
    
    # VRAM FIX: Batch size 1. 
    # With 2 GPUs and 8 accum steps, your effective batch size is still 16!
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8, 
    
    learning_rate=2e-4,
    # fp16=True,  # T4s require FP16
    # bf16=False, # T4s do NOT support BF16
    fp16=False,  
    bf16=True, 
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch", 
    save_total_limit=2, 
)

sft_trainer = SFTTrainer(
    model=model, 
    tokenizer=tokenizer, 
    train_dataset=sft_dataset, 
    args=sft_config
)

print("Starting SFT Phase 1 with Qwen2.5-Coder-3B...")
sft_trainer.train()

# Save the conditional adapters!
model.save_pretrained("./sft-lora-adapter-3B")
tokenizer.save_pretrained("./sft-lora-adapter-3B")

/mnt/c/Users/Josue/Documents/CS-GY-6923- Deep Learning/Midterm/unsloth_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/c/Users/Josue/Documents/CS-GY-6923- Deep Learning/Midterm/unsloth_env/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


Generating train split: 49043 examples [00:00, 78049.20 examples/s]


==((====))==  Unsloth 2026.3.10: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 3070 Ti. Num GPUs = 1. Max memory: 8.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 767.38it/s]
Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.3.10 patched 36 layers with 36 QKV layers, 36 O layers and 0 MLP layers.
Unsloth: Tokenizing ["text"] (num_proc=7): 100%|██████████| 25000/25000 [00:12<00:00, 2060.38 examples/s]


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting SFT Phase 1 with Qwen2.5-Coder-3B...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25,000 | Num Epochs = 1 | Total steps = 3,125
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 7,372,800 of 3,093,311,488 (0.24% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.711421
20,0.696054
30,0.727318
40,0.657541
50,0.686142
60,0.633067
70,0.617953
80,0.602682
90,0.587042
100,0.602635


('./sft-lora-adapter-3B/tokenizer_config.json',
 './sft-lora-adapter-3B/tokenizer.json')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Loading base model to CPU.")
# Load the base model strictly to the CPU to avoid the 8GB VRAM limit
base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Qwen2.5-Coder-3B", 
    torch_dtype=torch.float16,
    device_map="cpu", 
)

print("Attaching LoRA adapter.")
# Attach your Phase 1 adapter
model = PeftModel.from_pretrained(
    base_model,
    "./sft-lora-adapter-3B",
    device_map="cpu",
)

print("Merging weights (this will take a while).")
# Fuse the LoRA weights directly into the base model's linear layers
merged_model = model.merge_and_unload()

print("Saving merged model to disk.")
# Export the standalone Hugging Face model
merged_model.save_pretrained("./sft-merged-model-3b")

tokenizer = AutoTokenizer.from_pretrained("./sft-lora-adapter-3B")
tokenizer.save_pretrained("./sft-merged-model-3b")

print("Merge complete! Safe for Outlines inference/GRPO.")

/mnt/c/Users/Josue/Documents/CS-GY-6923- Deep Learning/Midterm/unsloth_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


Loading base model to CPU.


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:28<00:00, 15.22it/s]


Attaching LoRA adapter.
Merging weights (this will take a while).
Saving merged model to disk.


Writing model shards: 100%|██████████| 1/1 [04:44<00:00, 284.30s/it]


Merge complete! Safe for Outlines inference/GRPO.


In [ ]:
import gc

del model, base_model, merged_model, tokenizer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import os
import pandas as pd
import torch
import io
import cairosvg
import numpy as np
from PIL import Image
import xml.etree.ElementTree as ET
import warnings
import re

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, AutoModel, TextStreamer
import outlines
from outlines.types import CFG

warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

print("Initializing local inference.")
device = "cuda"

# ==========================================
# LOAD MAIN MODEL (16-BIT PRECISION)
# ==========================================
print("Loading 3B model in 16-bit (Float16).")
hf_model = AutoModelForCausalLM.from_pretrained(
    "./sft-merged-model-3b", 
    torch_dtype=torch.float16, # This replaces the 4-bit quantization!
    device_map=device, 
)

hf_model.generation_config.max_length = None
hf_tokenizer = AutoTokenizer.from_pretrained("./sft-merged-model-3b")

# The Outlines CFG guarantees 100% valid SVG formatting
generator = outlines.from_transformers(hf_model, hf_tokenizer)

official_kaggle_grammar = CFG("""
    ?start: WS? svg WS?
    svg: "<svg" WS? ATTR_LIST ">" WS? elements "</svg>"
    elements: (element | TEXT)*
    element: "<" TAG WS? ATTR_LIST "/>" WS? | "<" TAG WS? ATTR_LIST ">" WS? elements "</" TAG ">" WS?
    TAG: "svg" | "g" | "path" | "rect" | "circle" | "ellipse" | "line" | "polyline" | "polygon" | "defs" | "use" | "symbol" | "clipPath" | "mask" | "linearGradient" | "radialGradient" | "stop" | "text" | "tspan" | "title" | "desc" | "style" | "pattern" | "marker" | "filter"
    ATTR_LIST: (/[a-zA-Z0-9_:-]+/ WS? "=" WS? /"[^"]*"/ WS?)*
    TEXT: /[^<]+/
    WS: /[ \\t\\n\\r]+/
""")

# ==========================================
# LOAD SIGLIP JUDGE (16-BIT PRECISION)
# ==========================================
print("Loading SigLIP Judge.")
siglip_id = "google/siglip-so400m-patch14-384"
processor = AutoProcessor.from_pretrained(siglip_id)

judge = AutoModel.from_pretrained(
    siglip_id, 
    torch_dtype=torch.float16 # Crucial for saving VRAM
).to(device)
judge.eval() 


/mnt/c/Users/Josue/Documents/CS-GY-6923- Deep Learning/Midterm/unsloth_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
`torch_dtype` is deprecated! Use `dtype` instead!


Initializing local inference.
Loading 3B model in 16-bit (Float16).


Loading weights: 100%|██████████| 434/434 [01:00<00:00,  7.14it/s]


Loading SigLIP Judge.


The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 888/888 [00:00<00:00, 1710.14it/s]


SiglipModel(
  (text_model): SiglipTextTransformer(
    (embeddings): SiglipTextEmbeddings(
      (token_embedding): Embedding(32000, 1152)
      (position_embedding): Embedding(64, 1152)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-26): 27 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (fc2): Linear(in_features=4304,

In [ ]:
# ==========================================
# HELPER FUNCTIONS
# ==========================================
def extract_svg(text):
    match = re.search(r'(<svg.*?</svg>)', text, re.IGNORECASE | re.DOTALL)
    return match.group(1) if match else text


def heal_svg(raw_svg):
    """Saves truncated SVGs by chopping off broken paths and closing the tag."""
    if raw_svg.strip().endswith("</svg>"):
        return raw_svg
        
    # Find the last perfectly closed element (like a completed <path />)
    last_closed_idx = raw_svg.rfind("/>")
    
    if last_closed_idx != -1:
        # Chop off the broken half-written path and seal the SVG
        healed_svg = raw_svg[:last_closed_idx + 2] + "\n</svg>"
        return healed_svg
        
    return raw_svg

def is_kaggle_compliant(svg_string):
    if len(svg_string) > 16000: return False
    try:
        root = ET.fromstring(svg_string)
        if root.tag.split('}')[-1] != 'svg': return False
    except ET.ParseError: return False
    if svg_string.count("<path") > 256: return False
    return True

def render_to_numpy(svg_string):
    try:
        png_data = cairosvg.svg2png(bytestring=svg_string.encode('utf-8'), output_width=256, output_height=256, background_color="white")
        return np.array(Image.open(io.BytesIO(png_data)).convert('L'))
    except: return None

def select_best_svg(prompt_text, candidate_svgs):
    valid_images, valid_svgs = [], []
    
    for i, raw_svg in enumerate(candidate_svgs):
        clean_svg = extract_svg(raw_svg)

        # INVESTIGATION PRINT
        if not is_kaggle_compliant(clean_svg):
            print(f"      [!] Candidate {i+1} failed compliance. Length: {len(clean_svg)}")
            print(f"Raw string: {clean_svg[-100:]}") # Uncomment to see how it ended
            continue
            
        img = render_to_numpy(clean_svg)
        if img is None:
            print(f"      [!] Candidate {i+1} failed CairoSVG render.")
            continue
            
        valid_images.append(Image.fromarray(img).convert('RGB'))
        valid_svgs.append(clean_svg) 
            
    if not valid_images: 
        print("      [X] ALL CANDIDATES FAILED. Returning empty fallback.")
        return "<svg></svg>"
        
    if len(valid_images) == 1: return valid_svgs[0]

    inputs = processor(
        text=[prompt_text], 
        images=valid_images, 
        return_tensors="pt", 
        padding="max_length", 
        truncation=True, 
        max_length=64    
    ).to(device)
    
    with torch.no_grad():
        outputs = judge(**inputs)
        scores = outputs.logits_per_image.squeeze().cpu().numpy()
        
    if scores.ndim == 0:
        return valid_svgs[0]
        
    return valid_svgs[scores.argmax()]

def select_best_svg_healed(prompt_text, candidate_svgs):
    valid_images, valid_svgs = [], []
    
    for i, raw_svg in enumerate(candidate_svgs):
        clean_svg = extract_svg(raw_svg)

        # HEAL TRUNCATED STRINGS!
        healed_svg = heal_svg(clean_svg)
        
        # INVESTIGATION PRINT
        if not is_kaggle_compliant(healed_svg):
            print(f"      [!] Candidate {i+1} failed compliance. Length: {len(healed_svg)}")
            print(f"Raw string: {healed_svg[-100:]}") # Uncomment to see how it ended
            continue
            
        img = render_to_numpy(healed_svg)
        if img is None:
            print(f"      [!] Candidate {i+1} failed CairoSVG render.")
            continue
            
        valid_images.append(Image.fromarray(img).convert('RGB'))
        valid_svgs.append(healed_svg) 
            
    if not valid_images: 
        print("      [X] ALL CANDIDATES FAILED. Returning empty fallback.")
        return "<svg></svg>"
        
    if len(valid_images) == 1: return valid_svgs[0]

    inputs = processor(
        text=[prompt_text], 
        images=valid_images, 
        return_tensors="pt", 
        padding="max_length", 
        truncation=True, 
        max_length=64    
    ).to(device)
    
    with torch.no_grad():
        outputs = judge(**inputs)
        scores = outputs.logits_per_image.squeeze().cpu().numpy()
        
    if scores.ndim == 0:
        return valid_svgs[0]
        
    return valid_svgs[scores.argmax()]

The below process took 17 hours and 14 minutes for 1000 samples.

In [ ]:
# ==========================================
# INFERENCE LOOP
# ==========================================
if __name__ == "__main__":
    test_df = pd.read_csv("dl-spring-2026-svg-generation/test.csv") 
    csv_filename = "./submission-3b.csv"
    resume_index = 0
    
    # We will generate 2 candidates per prompt to feed to SigLIP
    NUM_CANDIDATES = 2

    MAX_CONTEXT_WINDOW = 1536
    
    print(f"\nStarting Inference on {len(test_df)} prompts...")

    for idx in range(resume_index, len(test_df)):
        row = test_df.iloc[idx]
        print(f"\n[{idx+1}/{len(test_df)}] Generating ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        open_comment = "<!-" + "-"
        close_comment = "-" + "->"
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        
        # Initialize the streamer here so it's fresh for each prompt
        streamer = TextStreamer(hf_tokenizer, skip_prompt=True, skip_special_tokens=True)
        
        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        candidates = [] 
        
        # ⚠️ SEQUENTIAL GENERATION LOOP (Saves VRAM!) ⚠️
        # for i in range(NUM_CANDIDATES):
        # print(f"  -> Generating 4 Candidate {i+1}/{NUM_CANDIDATES}...")
        print(f"  ~ Generating {NUM_CANDIDATES} Candidates...")
        try:
            # We ask outlines to generate exactly ONE sequence at a time
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                # max_new_tokens=1000,
                # The Safe Limit: Respects your 1536 SFT training boundary
                # max_new_tokens=1400,
                # max_new_tokens=1536,
                # max_new_tokens=1700,
                # max_new_tokens=1850,
                # max_new_tokens=2048,
                max_new_tokens=safe_max_new_tokens,
                num_return_sequences=NUM_CANDIDATES, 
                # streamer=streamer # Watch it build the SVG live!
            )
            
            # Outlines might return a list of 1 string or just a string, handle both safely
            if isinstance(raw_candidate, list):
                for candidate in raw_candidate:
                    candidates.append(candidate)
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"    Generation failed for 2 candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        best_svg = select_best_svg(row["prompt"], candidates)
        print(best_svg[:200])  # Print the first 200 chars of the best SVG for sanity check
        
        result_df = pd.DataFrame([{"id": row["id"], "svg": best_svg}])
        if idx == 0 or not os.path.exists(csv_filename):
            result_df.to_csv(csv_filename, index=False, mode='w')
        else:
            result_df.to_csv(csv_filename, index=False, mode='a', header=False)
            
        # Clear cache before moving to the next totally new prompt
        torch.cuda.empty_cache()

    print("\nLocal inference complete! Check submission-3b.csv")


Starting Inference on 1000 prompts...

[1/1000] Generating ID: fa1d8fa7-080f-4269-a9cf-a17562c9a0ca | Prompt: firewood stack cut logs wood with leaf i...
  ~ Prompt tokens: 17 | Tokens available for SVG: 1509
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate 2 failed compliance. Length: 1687
Raw string: 392 148.8,392 Z"/><path fill="#131415" d="M148.8,384 C158.4,384 168,374.4 168,364.8 C168,354.4 158.4
<svg xmlns="http://www.w3.org/2000/svg" height="1024" viewBox="0 0 1024 1024" width="1024"><defs/><path d="M248,384 L248,768 L576,768 L576,384 Z M32,111 L111,32 L111,384 L289,384 L32,111 Z M289,753 L3

[2/1000] Generating ID: 6eede943219547c22ac56085027d33cc | Prompt: The image shows five horizontal lines of...
  ~ Prompt tokens: 27 | Tokens available for SVG: 1499
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200px" width="200px">

The below process took 10 Hours and 48 Minutes for 366 Samples

In [ ]:
import pandas as pd
import warnings

# Silence warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

# Identify the missing rows from Pass 1
sub_df = pd.read_csv("./submission-3b.csv")
test_df = pd.read_csv("dl-spring-2026-svg-generation/test.csv")

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs. Starting Rescue Run.")

NUM_CANDIDATES = 2

# If there are no blanks, we can safely exit!
if len(rescue_df) == 0:
    print("No blanks found!")
    sub_df.to_csv("./submission_rescued-3b.csv", index=False) 
else:
    # The 2048-Token Rescue Loop
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"

    for idx, row in rescue_df.iterrows():
        print(f"Rescuing ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        # The Critical Prompt Injection 
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        candidates = [] 

        # GENERATION LOOP
        print(f"  ~ Generating {NUM_CANDIDATES} Candidates...")
        try:
            # THE KEY DIFFERENCE: max_new_tokens bumped to 2048 to prevent the guillotine
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                max_new_tokens=2048,
                num_return_sequences=NUM_CANDIDATES, 
                # streamer=streamer # Watch it build the SVG live!
            )
            
            # Outlines might return a list of 1 string or just a string, handle both safely
            if isinstance(raw_candidate, list):
                for candidate in raw_candidate:
                    candidates.append(candidate)
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        
        # We use the robust select_best_svg_healed from Pass 1 
        # (It already includes the is_kaggle_compliant check and CLIP scoring)
        best_svg = select_best_svg_healed(row["prompt"], candidates)
        print(best_svg[-200:])  # Print the last 200 chars of the best SVG for sanity check

        # Inject the rescued SVG back into the main dataframe in memory
        sub_df.loc[sub_df['id'] == row['id'], 'svg'] = best_svg
        
        # Save incrementally to the final rescue file
        sub_df.to_csv("./submission_rescued-3b.csv", index=False)

        # Clear cache before moving to the next totally new prompt
        torch.cuda.empty_cache()

    print("\nRescue complete! Final 'submission_rescued-3b.csv' is ready.")

Found 366 blank SVGs. Starting Rescue Run.
Rescuing ID: ea045c7a247166f061ce504d9b7ccaab | Prompt: A stylized icon depicting a curved arrow...
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
L12,-6 L12,-4 L10,-4 L10,-6 Z M8,-6 L10,-6 L10,-4 L8,-4 L8,-6 Z M6,-6 L8,-6 L8,-4 L6,-4 L6,-6 Z M4,-6 L6,-6 L6,-4 L4,-4 L4,-6 Z M2,-6 L4,-6 L4,-4 L2,-4 L2,-6 Z M0,-6 L2,-6 L2,-4 L0,-4 L0,-6 Z"/></svg>
Rescuing ID: 8fe82f3af89e487b31236ca829c3f071 | Prompt: The image contains black geometric shape...
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200px" width="200px"><defs/>
</svg>
Rescuing ID: 625181a2-600d-4db5-83c6-1a34676eb0dc | Prompt: The image is a black and white represent...
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200px" width="200px"><defs/>
<

In [6]:
# Valid SVG checks

# 1. Identify the rows with char > 16000 (This includes blanks and any other invalid outputs that slipped through)
sub_df['svg'] = sub_df['svg'].fillna("<svg></svg>")
oversized_mask = sub_df['svg'].str.len() > 16000
oversized_ids = sub_df[oversized_mask]['id'].tolist()
fix_df = test_df[test_df['id'].isin(oversized_ids)]

print(f"Found {len(fix_df)} SVGs exceeding 16,000 characters.")

# 2. Identify the blank SVGs
missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs.")

Found 0 SVGs exceeding 16,000 characters.
Found 0 blank SVGs.
